In [ ]:
import pandas as pd
import os

base_path = '/content/drive/MyDrive/FINAL PHISHING DATASET'

phish_csv = os.path.join(base_path, 'Phishing (1).csv')
legit_csv = os.path.join(base_path, 'Legitimate.csv')

phish_img_dir = os.path.join(base_path, 'PHISHING')
legit_img_dir = os.path.join(base_path, 'LEGITIMATE')

phish_df = pd.read_csv(phish_csv)
legit_df = pd.read_csv(legit_csv)

print("Phishing CSV Columns:", phish_df.columns)
print("Legitimate CSV Columns:", legit_df.columns)

def sort_numerically(file_list):
    return sorted(file_list, key=lambda x: int(''.join(filter(str.isdigit, x)) or 0))

phish_images = sort_numerically(os.listdir(phish_img_dir))
legit_images = sort_numerically(os.listdir(legit_img_dir))

print(f"Phishing images: {len(phish_images)}, Phishing CSV: {len(phish_df)}")
print(f"Legitimate images: {len(legit_images)}, Legitimate CSV: {len(legit_df)}")

def map_images_to_urls(df, images, img_dir):
    image_paths = []
    for i in range(len(images)):
        csv_index = i + 1
        if csv_index < len(df):
            image_paths.append(os.path.join(img_dir, images[i]))
        else:
            image_paths.append(os.path.join(img_dir, images[-1]))
    df = df.iloc[:len(image_paths)].copy()
    df['image_path'] = image_paths
    return df

phish_df = map_images_to_urls(phish_df, phish_images, phish_img_dir)
legit_df = map_images_to_urls(legit_df, legit_images, legit_img_dir)

data = pd.concat([phish_df, legit_df], ignore_index=True)

print("\n Combined dataset preview:")
print(data.head())


Phishing CSV Columns: Index(['URL', 'Label'], dtype='object')
Legitimate CSV Columns: Index(['URL', 'Label'], dtype='object')
Phishing images: 3000, Phishing CSV: 3000
Legitimate images: 3000, Legitimate CSV: 3000

 Combined dataset preview:
                             URL     Label  \
0      https://acessar-teem.com/  Phishing   
1  https://www.saqueagorabr.cfd/  Phishing   
2          https://jetdicey.com/  Phishing   
3       https://marty-drops.xyz/  Phishing   
4         https://jentcas.com/w/  Phishing   

                                          image_path  
0  /content/drive/MyDrive/FINAL PHISHING DATASET/...  
1  /content/drive/MyDrive/FINAL PHISHING DATASET/...  
2  /content/drive/MyDrive/FINAL PHISHING DATASET/...  
3  /content/drive/MyDrive/FINAL PHISHING DATASET/...  
4  /content/drive/MyDrive/FINAL PHISHING DATASET/...  


In [ ]:
phish_df['label'] = 1
legit_df['label'] = 0

data = pd.concat([phish_df, legit_df], ignore_index=True)

url_col = 'url' if 'url' in data.columns else data.columns[0]
data['url'] = data[url_col].astype(str).str.strip()

print(data[['url', 'image_path', 'label']].head())


                             url  \
0      https://acessar-teem.com/   
1  https://www.saqueagorabr.cfd/   
2          https://jetdicey.com/   
3       https://marty-drops.xyz/   
4         https://jentcas.com/w/   

                                          image_path  label  
0  /content/drive/MyDrive/FINAL PHISHING DATASET/...      1  
1  /content/drive/MyDrive/FINAL PHISHING DATASET/...      1  
2  /content/drive/MyDrive/FINAL PHISHING DATASET/...      1  
3  /content/drive/MyDrive/FINAL PHISHING DATASET/...      1  
4  /content/drive/MyDrive/FINAL PHISHING DATASET/...      1  


In [ ]:
import re
import numpy as np
from tqdm import tqdm

def extract_url_features(url):
    return {
        "url_length": len(url),
        "count_dot": url.count('.'),
        "count_hyphen": url.count('-'),
        "count_at": url.count('@'),
        "count_digit": sum(c.isdigit() for c in url),
        "has_https": int(url.startswith('https')),
        "has_ip": int(bool(re.match(r'http[s]?://\d+\.\d+\.\d+\.\d+', url))),
        "has_suspicious_word": int(any(w in url for w in ['login','secure','update','verify','signin','bank']))
    }

url_features = [extract_url_features(u) for u in tqdm(data['url'])]
url_feat_df = pd.DataFrame(url_features)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
url_feat_scaled = scaler.fit_transform(url_feat_df)

print("URL feature shape:", url_feat_scaled.shape)


100%|██████████| 6000/6000 [00:00<00:00, 26150.25it/s]


URL feature shape: (6000, 8)


In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df, train_url_feat, test_url_feat = train_test_split(
    data[['image_path', 'label']], url_feat_scaled, test_size=0.2, random_state=42, stratify=data['label']
)

print(len(train_df), len(test_df))


4800 1200


In [ ]:
from transformers import ViTImageProcessor
from PIL import Image
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vit_processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

In [ ]:
from torch.utils.data import Dataset

class PhishDataset(Dataset):
    def __init__(self, df, url_features, processor):
        self.df = df.reset_index(drop=True)
        self.url_features = torch.tensor(url_features, dtype=torch.float32)
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        label = torch.tensor(self.df.loc[idx, 'label'], dtype=torch.float32)
        url_feat = self.url_features[idx]

        img = Image.open(img_path).convert('RGB')
        inputs = self.processor(img, return_tensors="pt")
        pixel_values = inputs['pixel_values'].squeeze(0)

        return pixel_values, url_feat, label


In [ ]:

from transformers import ViTModel
import torch.nn as nn

class ViTPhishFusion(nn.Module):
    def __init__(self, url_dim):
        super().__init__()
        self.vit = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
        self.vit.requires_grad_(False)  # freeze ViT
        self.fc = nn.Sequential(
            nn.Linear(768 + url_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, images, url_feats):
        vit_outputs = self.vit(images)
        img_feat = vit_outputs.last_hidden_state[:, 0, :]  # [CLS] token
        combined = torch.cat((img_feat, url_feats), dim=1)
        return self.fc(combined)


In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim

train_dataset = PhishDataset(train_df, train_url_feat, vit_processor)
test_dataset  = PhishDataset(test_df,  test_url_feat,  vit_processor)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=16)

model = ViTPhishFusion(url_dim=train_url_feat.shape[1]).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-4)


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

In [ ]:
import os
import torch

num_epochs = 40
batch_size = 64
checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, num_workers=4, pin_memory=True)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

start_epoch = 0
checkpoint_files = sorted([f for f in os.listdir(checkpoint_dir) if f.endswith(".pth")])
if checkpoint_files:
    latest_checkpoint = os.path.join(checkpoint_dir, checkpoint_files[-1])
    print(f"🔄 Loading checkpoint {latest_checkpoint} ...")
    checkpoint = torch.load(latest_checkpoint, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch']
    print(f"Resumed from epoch {start_epoch}")
else:
    print("🚀 Starting training from scratch...")

for epoch in range(start_epoch, num_epochs):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for imgs, url_feats, labels in train_loader:
        imgs, url_feats, labels = imgs.to(device), url_feats.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs, url_feats).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        predicted = (outputs > 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total

    model.eval()
    test_loss = 0
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for imgs, url_feats, labels in test_loader:
            imgs, url_feats, labels = imgs.to(device), url_feats.to(device), labels.to(device)
            outputs = model(imgs, url_feats).squeeze()
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            predicted = (outputs > 0.5).float()
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()

    test_acc = 100 * test_correct / test_total

    scheduler.step()

    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | "
          f"Train Acc: {train_acc:.2f}% | "
          f"Test Loss: {test_loss/len(test_loader):.4f} | "
          f"Test Acc: {test_acc:.2f}%")

    if (epoch + 1) % 5 == 0:
        checkpoint_path = os.path.join(checkpoint_dir, f"model_epoch_{epoch+1}.pth")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
        }, checkpoint_path)
        print(f"Checkpoint saved at {checkpoint_path}")

        checkpoint_files = sorted([f for f in os.listdir(checkpoint_dir) if f.endswith(".pth")])
        if len(checkpoint_files) > 5:
            os.remove(os.path.join(checkpoint_dir, checkpoint_files[0]))


🚀 Starting training from scratch...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch [1/40] | Train Loss: 0.6527 | Train Acc: 67.52% | Test Loss: 0.5898 | Test Acc: 72.83%


KeyboardInterrupt: 